> **💾 First: save your own copy** · This notebook opens from the course website, so Colab does not keep your changes. Before you do anything else, go to **File → Save a copy in Drive**. Colab opens your copy in a new tab: work in that tab and close this one. You find your copy later in Google Drive, in the folder *Colab Notebooks*.

![Cultural Data Analysis · Week 4: Cultural Data Forms: Text](https://goto4711.github.io/cda-teaching-tools/assets/notebooks/week4-banner.png)

# Frozen in Time: Can LLMs Forecast World Events?

In this workshop, you learn how researchers talk to language models through an **API**, and you test a claim from the news: that AI models can predict the future.

A language model learns nothing after its training ends; it is *frozen in time*. In the ABC News video [Can AI predict the future?](https://www.youtube.com/watch?v=Oz7GXguhCfA), journalist Matt Bevan turns this into an experiment. He gave three models, whose training ended before the 2026 war with Iran, a simulated battle plan and intelligence dossier as on 27 February 2026, and asked how likely Trump's war aims were. Looking back, he found their predictions "spookily good": 12 of 14 correct. On other topics they did badly, which he explains with *recency bias*: the models assume that what usually happens will happen again.

We test this more carefully: with several runs per model, probabilities instead of yes/no, a simple baseline, and a check whether a model already knows the answer.

> **⚠️ Important** · The questions are about real wars in which many people died. We study how models forecast and whose expertise they repeat, not who "wins".

| Part | What you do | Minutes |
|---|---|---|
| 1. Talking to a model through its API | build a request, read the answer, count tokens and costs | 15 |
| 2. Where does a model's knowledge stop? | probe the models with dated events | 8 |
| 3. The forecasts | read a briefing and the models' probabilities | 4 |
| 4. Scoring the forecasts | Brier score, a "nothing changes" baseline, leaks | 10 |
| 5. How do the models explain themselves? | a dictionary method and tf-idf from the lecture on the models' reasons | 6 |
| 6. One answer is one sample | how much do the runs differ? | 4 |
| Reflection | three short answers | 3 |

> **⚠️ Important** · **This is a group exercise.** Work in groups of three or four. Your group chooses **one** of the two Week 4 notebooks: *Are All LLMs the Same?* (whose history do language models tell?) or *Frozen in Time* (can language models forecast world events?). Everyone runs the notebook on their own laptop, but you work through it together. Stop at the **💬 Discuss** boxes and talk before you go on. One API key per group is enough. Each member hands in their own notebook.

**Group members:** *write the names of your group here*

> **⚠️ Important** · You don't need an API key: without one, the notebook uses answers that we collected for you with exactly the same requests. With a free key you can send the requests yourself (see Part 1). Getting a key takes about five minutes, so do it before the workshop or at home afterwards.

Please run the cell below to set up the environment.

In [ ]:
#Keep cell
import os, re, json, random, urllib.request
from collections import Counter
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
for r in ['punkt', 'punkt_tab', 'stopwords']:
  nltk.download(r, quiet=True)
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

try:
  import learntools.core
  from learntools.core import binder
  binder.bind(globals())
except:
  !pip install -q git+https://github.com/Kaggle/learntools.git
  import learntools.core
  from learntools.core import binder
  binder.bind(globals())

BASE = 'https://raw.githubusercontent.com/goto4711/cdai/main/'
for f in ['prompting_workshop_tests.py', 'forecasting_workshop_tests.py', 'llm_dates_2026.json', 'llm_forecasts_2026.json']:
  if not os.path.exists(f):
    urllib.request.urlretrieve(BASE + f, f)

from prompting_workshop_tests import *
from forecasting_workshop_tests import *

data = json.load(open('llm_dates_2026.json', encoding='utf-8'))          # for Part 1
responses = pd.DataFrame(data['responses'])
fdata = json.load(open('llm_forecasts_2026.json', encoding='utf-8'))     # for Parts 2-5
print(f"Forecasts collected on {fdata['about']['collected']} from {len(fdata['about']['models'])} models")

---

## 1. Talking to a model through its API (15 min)

An **API** (application programming interface) is how one program talks to another. For a language model, your program sends a **request** over the internet and gets a **response** back. The chat window of ChatGPT or Gemini does exactly this behind the scenes.

Researchers use the API rather than the chat window because they can

* send **exactly the same prompt** many times, and to several models;
* set and record the **settings**, such as the model version and the *temperature* (how much randomness);
* **save** every answer with its details, so others can check the work.

Note that the API is not the same as the chat app: the app adds its own instructions, memory and sometimes a web search. Through the API we get closer to the model itself (though the provider still filters and processes requests on its side).

A request has three parts:

1. a **URL**: the address of the model;
2. **headers**: who you are, i.e. your **API key**;
3. a **body**: what you send, i.e. the prompt and the settings, written as JSON (nested Python dictionaries and lists).

> **💡 Hint** · **Optional: your own free key.** Go to [aistudio.google.com](https://aistudio.google.com) → *Get API key*. In Colab, click the key symbol on the left → *Add new secret*, name it `GEMINI_API_KEY`, paste the key and switch on *Notebook access*. Read the terms of the free tier: Google may use what you send to improve its products, so never send personal data.

> **⚠️ Important** · Never paste an API key into your code. Anyone who sees the notebook can then use it, at your cost. Notebooks are shared all the time.

Run the next cell. It looks for a key in Colab's secrets.

In [ ]:
#Keep cell
try:
  from google.colab import userdata
  GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
  GEMINI_API_KEY = None

if GEMINI_API_KEY:
  print('Key found: this notebook will ask Gemini itself.')
else:
  print('No key: this notebook uses the pre-collected answers. Everything works the same.')

Now we build the request. Run the next cell to set the URL and the headers. Note how the key goes into the headers, not into the URL or the code.

In [ ]:
#Keep cell
PROMPT = data['about']['prompts']['en']
MODEL = data['about'].get('student_model', 'gemini-3.5-flash-lite')   # a small Gemini model with a free tier

url = f'https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent'
headers = {'Content-Type': 'application/json',
           'x-goog-api-key': GEMINI_API_KEY}

print('URL:', url)
print('\nPrompt:\n' + PROMPT)

The **body** is a nested dictionary. Gemini expects the prompt as `text` inside `parts` inside `contents`, and the settings in `generationConfig`.

Replace the two `____` below: put our `PROMPT` in the right place, and set the temperature to `1.0`. The **temperature** controls the randomness when the model picks the next word: low values make it pick the most likely words, `1.0` (the usual default) lets it vary. Google recommends keeping Gemini 3 models at `1.0`.

Complete the next cell.

In [ ]:
#Complete cell
body = {
    'contents': [{'parts': [{'text': ____}]}],
    'generationConfig': {'temperature': ____}
}

In [ ]:
q1.check()
#q1.hint()
#q1.solution()

Now we send the request with `requests.post()`. If you have no key, the cell takes a saved response to exactly this request from our data.

Run the next cell. The response is long, so we only print the start.

In [ ]:
#Keep cell
response = data['example_response']        # the saved answer, used if there is no key or the request fails
if GEMINI_API_KEY:
  r = requests.post(url, headers=headers, json=body, timeout=120)
  print('Status code:', r.status_code)
  if r.status_code == 200:
    response = r.json()
  else:
    print('Error message:', r.json().get('error', {}).get('message'), '\nWe use the saved answer instead.\n')

print(json.dumps(response, indent=2, ensure_ascii=False)[:1500])

The **status code** tells you whether it worked:

| Code | Meaning |
|---|---|
| `200` | OK |
| `400` | a mistake in the request (with Gemini also: a wrong key) |
| `403` | not allowed, e.g. a problem with the key |
| `404` | not found, e.g. a model name that does not exist (any more) |
| `429` | too many requests: wait a minute |
| `503` | the service is overloaded: try again later |

The response is JSON again: dictionaries inside lists inside dictionaries.

Find the model's answer in `response` and save it in `answer`. Follow the path in the printout: the key `'candidates'` holds a list; take its first item (`[0]`), then `'content'`, then `'parts'`, again the first item, then `'text'`.

Complete the next cell.

In [ ]:
#Complete cell
answer = ____
print(answer)

In [ ]:
q2.check()
#q2.hint()
#q2.solution()

Every request costs money, counted in **tokens**: pieces of words. In English, a token is roughly ¾ of a word; Dutch and especially Chinese need more tokens for the same content, so the same request costs more. The response says in `usageMetadata` how many tokens went in (your prompt) and came out. Output tokens cost more than input tokens. Models that "think" before answering also bill the thinking as output (`thoughtsTokenCount`).

Save the number of input tokens (key `'promptTokenCount'`) in `tokens_in` and the tokens of the answer (key `'candidatesTokenCount'`) in `tokens_answer`. The cell then adds the thinking tokens and computes the cost in dollars. The prices are per million tokens.

Complete the next cell.

In [ ]:
#Complete cell
PRICE_IN = 0.30    # dollars per 1 million input tokens (Gemini 3.5 Flash-Lite, October 2026)
PRICE_OUT = 2.50   # dollars per 1 million output tokens, thinking included

usage = response['usageMetadata']
tokens_in = ____
tokens_answer = ____
tokens_out = tokens_answer + usage.get('thoughtsTokenCount', 0)
cost = (tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1_000_000

print(f'{tokens_in} tokens in, {tokens_out} tokens out: ${cost:.5f}')
print(f'Our data set has 150 such answers: about ${150 * cost:.2f}')

In [ ]:
q3.check()
#q3.hint()
#q3.solution()

**Tokens are not words.** In the lecture, you split texts into words with NLTK's `word_tokenize()`. Language models use their own tokenizer, which cuts rare words into smaller pieces (*sub-words*) and turns each piece into a number. Run the next cell to compare the two counts for the same answer.

In [ ]:
#Keep cell
words = word_tokenize(answer)
print(len(words), 'words and punctuation marks with word_tokenize(), as in the lecture')
print(tokens_answer, 'tokens billed by the API for the same answer')
print('First words:', words[:12])

Cheap for one experiment, but it adds up when you send thousands of texts.

Let's put the request into a function, `ask()`, so we can reuse it. Then we ask twice with the same prompt. Without a key, you get two of the pre-collected Gemini answers.

Run the next cell.

In [ ]:
#Keep cell
def ask(prompt, temperature=1.0):
  """Send one prompt to Gemini and return the text of the answer."""
  body = {'contents': [{'parts': [{'text': prompt}]}],
          'generationConfig': {'temperature': temperature}}
  r = requests.post(url, headers=headers, json=body, timeout=120)
  if r.status_code != 200:
    raise RuntimeError(f"{r.status_code}: {r.json().get('error', {}).get('message')}")
  parts = r.json()['candidates'][0]['content']['parts']
  return ''.join(p.get('text', '') for p in parts)   # the answer can come in several parts

gem = responses[(responses['label'] == 'Gemini') & (responses['language'] == 'en')]
first, second = gem['text'].iloc[0], gem['text'].iloc[1]     # two saved answers
if GEMINI_API_KEY:
  try:
    first, second = ask(PROMPT), ask(PROMPT)
  except Exception as e:
    print('The request did not work:', e, '- we use two saved answers instead.\n')

years_first = set(re.findall(r'\d{3,4}', first))
years_second = set(re.findall(r'\d{3,4}', second))
print('In both answers:     ', sorted(years_first & years_second))
print('Only in the first:   ', sorted(years_first - years_second))
print('Only in the second:  ', sorted(years_second - years_first))

Same model, same prompt, different lists. Why? And what does this mean for research that uses one answer from a chatbot?

> **💬 Discuss** · If someone in your group has a key, compare your lists: does each of you get a different answer from the same model? Together, decide what one answer from a chatbot is worth as evidence.

Work through the next cell to reveal the answer.

In [ ]:
q4.hint()
#q4.solution()

> **💡 Remember** · Other providers work the same way: a URL, a key in the headers, a JSON body with the prompt and settings, a JSON response. Only the names differ. An API has another property that matters for this workshop: **it does not search the web** unless you switch that on. Through the API, you talk to the model as it was when its training ended.

---

## 2. Where does a model's knowledge stop? (8 min)

To test forecasts, the model must not already know what happened. Models are "frozen" at their **training cutoff**, but providers don't always say exactly when that is, and models are often unsure themselves.

So we **probe**: we asked six models, through the API and without web search, about ten events whose dates we know, from the US election of November 2024 to the new Supreme Leader of Iran in March 2026. The answers were checked automatically for a key word (e.g. "Leo" for the current Pope).

Run the next cell to load the data and see which events each model knows.

In [ ]:
#Keep cell
probes = pd.DataFrame(fdata['probes'])
probes['event'] = probes['event_date'] + ' ' + probes['probe']
table = pd.crosstab(probes['label'], probes['event'], values=probes['knows'], aggfunc='max')
print('Models and the events they know (True), in date order:\n')
print(table[sorted(table.columns)].map(lambda known: 'knows' if known else '-').to_string())
print()
for m in fdata['about']['models']:
  print(f"{m['label']:17} = {m['model']:22} says its knowledge reaches: {m['self_cutoff']}")

Which is the **latest** event that each model knows? Select the rows of `probes` where the column `knows` is `True`, group them by model (`label`), and take the maximum of `event_date`. Dates written as `2025-06-22` can be compared like text: later dates are "larger".

Complete the next cell.

In [ ]:
#Complete cell
latest = ____
latest

In [ ]:
q12.check()
#q12.hint()
#q12.solution()

> **💬 Discuss** · Compare `latest` with what the models say about themselves. Do they agree? Split the probe answers among you, read them (e.g. `probes[probes['probe'] == 'P7'][['label', 'answer']]`) and check the automatic "knows" column together.

A model that knows nothing from our list (it is missing from `latest`) stopped learning before November 2024. The automatic check can be wrong both ways: a model might guess "Trump" for the 2024 election without knowing, or know an event but describe it without our key word.

**Is the free model from Part 1 frozen early enough?** With a key, the next cell asks your Gemini model two of the probe questions. Without a key, it shows what the models in our data answered.

In [ ]:
#Keep cell
check_probes = [p for p in fdata['freeze_probes'] if p['id'] in ('P10', 'P7')]
for p in check_probes:
  print(p['question'])
  if GEMINI_API_KEY:
    try:
      print('  Your model:', ask(fdata['about']['probe_instruction'] + '\n\n' + p['question']).strip()[:300])
    except Exception as e:
      print('  The request did not work:', e)
  else:
    newest = probes[(probes['probe'] == p['id'])].sort_values('label')
    for _, row in newest.iterrows():
      print(f"  {row['label']:17} {row['answer'][:150]}")
  print()

A model that knows what happened after a briefing's date cannot *forecast* it: it *remembers* it. We call this a **leak**. The rule: if a model knows any probe event from the briefing's date or later, its forecasts for that briefing are marked as leaks.

Run the next cell. It joins the forecasts with the questions and their outcomes, and marks the leaks.

In [ ]:
#Keep cell
questions = pd.DataFrame(fdata['questions'])
forecasts = pd.DataFrame(fdata['forecasts']).merge(
    questions[['id', 'briefing', 'question', 'answer', 'status_quo']], on=['id', 'briefing'])
forecasts['outcome'] = (forecasts['answer'] == 'yes').astype(int)      # 1 = it happened, 0 = it did not

as_of = {k: b['as_of'] for k, b in fdata['briefings'].items()}
def is_leak(row):
  known = probes[(probes['label'] == row['label']) & probes['knows']]
  return bool((known['event_date'] >= as_of[row['briefing']]).any())

forecasts['leak'] = forecasts.apply(is_leak, axis=1)
print('Forecasts per model (True = leak):')
print(pd.crosstab(forecasts['label'], forecasts['leak']))

---

## 3. The forecasts (4 min)

Each model received a **briefing** written as on a certain date, with only what was public then, and the questions for that briefing. It gave the probability (0 to 1) that the answer is *yes*, ten times each. Here is the briefing for the start of the Iran war, as in the video. Run the next cell.

In [ ]:
#Keep cell
b = fdata['briefings']['iran2026']
print(f"As of {b['as_of']}: {b['title']}\n")
print(b['text'], '\n')
print(fdata['about']['instructions_to_model'].replace('{as_of}', b['as_of']))

> **💬 Discuss** · Before you look at the table: each of you writes down your own probability for two or three of the Iran questions, as if it were 27 February 2026. Then compare with the models. Who in your group would have forecast better?

Run the next cell to see the average probability that each model gave, next to what actually happened. Leaked forecasts are left out.

In [ ]:
#Keep cell
iran = forecasts[(forecasts['briefing'] == 'iran2026') & ~forecasts['leak']]
view = iran.pivot_table(index=['id', 'question', 'answer'], columns='label', values='probability').round(2)
pd.set_option('display.max_colwidth', 70)
view

Read the table like the video does: which outcomes did the models see coming, and which surprised them? Then look at all briefings: `questions[['id', 'question', 'answer']]` lists every question.

---

## 4. Scoring the forecasts (10 min)

The video counted yes/no answers as right or wrong. But a model that says 55% and one that says 99% are not equally right. Forecasters therefore use the **Brier score**: the squared difference between the probability and the outcome (1 if it happened, 0 if not).

* 0 is perfect (100% for something that happened);
* 0.25 is what you get by always saying 50%;
* 1 is the worst (100% for something that did not happen).

Compute the Brier score of every forecast in a new column `brier`: subtract the `outcome` from the `probability` and square the result with `** 2`.

Complete the next cell.

In [ ]:
#Complete cell
forecasts['brier'] = ____
forecasts[['label', 'id', 'probability', 'outcome', 'brier']].head()

In [ ]:
q13.check()
#q13.hint()
#q13.solution()

Is a score good or bad? We need something to compare with: a **baseline**. We have two. "Always 50%" scores 0.25. "Nothing changes" gives 10% to every change and 90% to continuity (for example to "Khamenei is still Supreme Leader"). A model that cannot beat these has learned nothing useful about the situation.

Be careful with "nothing changes" here: our questions were chosen *after* big things had happened, so continuity often lost. Choosing questions with hindsight shapes every score, the models' too. Run the next cell.

In [ ]:
#Keep cell
forecasts['p_nothing_changes'] = forecasts['status_quo'].map({'yes': 0.9, 'no': 0.1}).fillna(0.5)
forecasts['brier_nothing_changes'] = (forecasts['p_nothing_changes'] - forecasts['outcome']) ** 2
forecasts['break'] = forecasts['answer'] != forecasts['status_quo']    # True: what happened broke with the status quo

Now leave out the leaks. Select the rows of `forecasts` where `leak` is `False` and save them in `fair`. (`~` means "not": `~forecasts['leak']` is `True` where `leak` is `False`.)

Complete the next cell.

In [ ]:
#Complete cell
fair = ____
print(len(fair), 'fair forecasts of', len(forecasts))

In [ ]:
q14.check()
#q14.hint()
#q14.solution()

Run the next cell to compare each model with the baselines, on the questions it could fairly answer, and with its own leaked forecasts.

In [ ]:
#Keep cell
scores = fair.groupby('label').agg(questions=('id', 'nunique'), model=('brier', 'mean'),
                                   nothing_changes=('brier_nothing_changes', 'mean'))
scores['always_50'] = 0.25
if forecasts['leak'].any():
  scores['leaked (remembered)'] = forecasts[forecasts['leak']].groupby('label')['brier'].mean()
else:
  print('No leaks in our data: every model could fairly forecast every question.')
print('Brier score: lower is better\n')
print(scores.round(3).to_string())

> **💬 Discuss** · Do the models beat "always 50%" and "nothing changes"? Why is "nothing changes" so bad here (think about how the questions were chosen)? Compare the leaked forecasts with the fair ones: what happens to the score when a model already knows the answer? Imagine a study that does not check for leaks: what would it have published? And look back at Part 2: would you have trusted the models' own statements about where their knowledge stops?

The video's explanation was **recency bias**: models expect continuity. If that is right, they should do worse on questions where what happened **broke with the status quo** (column `break`). Make a table with the average Brier score per model and per kind of question: group `fair` by `label` and by `break`. The `.unstack()` at the end turns the second group into columns.

Complete the next cell.

In [ ]:
#Complete cell
by_break = fair.groupby(['label', ____])['brier'].mean().unstack()
by_break.round(3)

In [ ]:
q15.check()
#q15.hint()
#q15.solution()

The video also found that the models did well on the Iran war, which their training on military analysis could explain. Run the next cell to compare the 2026 Iran war with the other events.

In [ ]:
#Keep cell
fair = fair.assign(topic=np.where(fair['briefing'].str.startswith('iran2026'), 'Iran war 2026', 'other events'))
fair.groupby(['label', 'topic'])['brier'].mean().unstack().round(3)

---

## 5. How do the models explain themselves? (6 min)

Every forecast comes with a one-sentence reason. If the video is right about recency bias, the models should argue with continuity: *historically*, *usually*, *unlikely*, *remain*. Let's test this with a method from the lecture.

In the lecture, `hate_score_()` counted how many words of a text appear in a hate-speech dictionary. We do the same with a small dictionary of **continuity words**. Run the next cell to see it, and a few reasons.

In [ ]:
#Keep cell
continuity_words = ['historically', 'historical', 'history', 'typically', 'usually', 'traditionally', 'precedent',
                    'unprecedented', 'unlikely', 'rarely', 'remain', 'remains', 'stable', 'resilient', 'entrenched',
                    'continue', 'continues', 'persist', 'survive', 'survived']
for _, row in fair.sample(4, random_state=1).iterrows():
  print(f"{row['label']} on {row['id']} ({row['probability']:.0%}): {row['reason']}")

Complete the function `continuity_score()`. It works like `hate_score_()` in the lecture: split the text into lower-case words with `word_tokenize()`, then add 1 for every word that is in `continuity_words`. Replace `____` with that condition.

Complete the next cell.

In [ ]:
#Complete cell
def continuity_score(text):
  score = 0
  words = [word.lower() for word in word_tokenize(text)]
  for word in words:
    if ____:
      score += 1
  return score

continuity_score('Historically, such regimes remain in power; a collapse is unlikely.')

In [ ]:
q16.check()
#q16.hint()
#q16.solution()

Run the next cell. It scores every reason and compares the questions where the status quo broke with the others. It also uses **tf-idf** from the lecture, with the same settings, to find the words that are typical for each model's reasons.

In [ ]:
#Keep cell
fair = fair.assign(continuity=fair['reason'].apply(continuity_score))
print('Average number of continuity words per reason:\n')
print(fair.groupby(['label', 'break'])['continuity'].mean().unstack().round(2).to_string())

reasons = fair.groupby('label')['reason'].apply(' '.join)
tfidf = TfidfVectorizer(lowercase = True, stop_words = 'english', token_pattern = '[a-zA-Z][a-zA-Z][a-zA-Z]+')
vec = tfidf.fit_transform(reasons)
tfidf_df = pd.DataFrame(vec.toarray().transpose(), index=tfidf.get_feature_names_out(), columns=reasons.index)
print('\nTypical words in each model\'s reasons (tf-idf):')
for m in tfidf_df.columns:
  print(f"  {m:17}", list(tfidf_df[m].sort_values(ascending=False).head(8).index))

Do the models argue with continuity more often for the questions where the status quo broke, which they often got wrong? Remember the lecture's question about the hate-speech dictionary: a word list depends on how people use language. What does our small list miss, and what does it count wrongly (*unlikely* can also argue for change)?

---

## 6. One answer is one sample (4 min)

The video asked each model **once**. We asked ten times. How much do the probabilities for the same question differ between runs? Compute the **standard deviation** of `probability` for each model and question: group `fair` by `label` and `id`, select `probability`, and use `.std()`.

Complete the next cell.

In [ ]:
#Complete cell
spread = ____
spread.sort_values(ascending=False).head(10).round(2)

In [ ]:
q17.check()
#q17.hint()
#q17.solution()

What does this mean for a yes/no count like "12 of 14 correct"? Run the next cell. It counts, for every single run, the share of questions a model got "right" (more than 50% for what happened, less than 50% for what did not), and shows the worst and the best run.

In [ ]:
#Keep cell
fair = fair.assign(right=(fair['probability'] > 0.5) == (fair['outcome'] == 1))
per_run = fair.groupby(['label', 'run'])['right'].mean().unstack()
print('Share of questions right in a single run:\n')
print(per_run.agg(['min', 'max'], axis=1).round(2).to_string())

---

## Reflection (3 min)

Discuss the three questions in your group, then each write your answers in two or three sentences.

1. **Can frozen models forecast world events?** Compare the models with the baselines, and the "break" questions with the others. Does your result support the video?
2. **The method:** the video used one run per model, 14 questions chosen after the outcomes were known, and a briefing written with hindsight. Which of these would change the result, and how would you design the test?
3. **Whose expertise?** The video says the models "know what the experts know" and have no "ideology to pursue (or at least it's not meant to)". Whose expertise is in the training data (think of the history exercise), and what would a "neutral" forecast be?

*Your answers*

> **⚠️ Important** · Run the next cell to create an automated HTML download of your notebook for submission to Canvas. Please **make sure you have saved the notebook** before running the code, and check the HTML file before you submit it.

In [ ]:
#Keep cell
import os

if not(os.path.exists('make_html.py')):
  print('Downloading make_html.py\n')
  !wget -q https://raw.githubusercontent.com/goto4711/cdai/refs/heads/main/make_html.py

from make_html import make_html

make_html("Week4_Forecasting.ipynb")